# Multi-triat Models
## Learning Objectives
- Fit Multi-trait models in sommer
- Identify scenarios in which multi-trait models provide increased accuracy compared to a univariate model.

## Load Packages

In [ ]:
#Loading libraries
req_packages<-c("ggplot2", "sommer")

for(i in c(1:length(req_packages))){
  if (!require(req_packages[i], character.only = TRUE)){
   install.packages(req_packages[i])
  }
}

## Analyze data using sommer
Here we read in and analyze the data using sommer.

**Reading the data**

In [ ]:
# reading the data and storing it as phenodat

phenodat=read.csv("https://raw.githubusercontent.com/Robbins-Lab/PLSCI-4170-7170/refs/heads/main/datasets/data_lab9/MTdataset_2025.csv",header=TRUE)

# Header = TRUE because the file has a header row in it.

#reading in the genomic relationship matrix
snpRelMat=read.csv("https://raw.githubusercontent.com/Robbins-Lab/PLSCI-4170-7170/refs/heads/main/datasets/data_lab9/GRM.csv",header=FALSE)
Gsnp=as.matrix(snpRelMat)+diag(.0005,length(snpRelMat[,1]),length(snpRelMat[,1]))

summary(phenodat)


**Creating a dataframe and running the sommer models**

In [ ]:
#creating data.frame()
library(sommer)

phenodf<-data.frame(Variety=as.factor(phenodat$Variety),Rep=as.factor(phenodat$Rep),P1=as.double(phenodat$Phenotype1), P2=as.double(phenodat$Phenotype2),P1NA=as.double(phenodat$PhenoMissing1), P2NA=as.double(phenodat$PhenoMissing2))

# Map elements in the relationship matrix to the phenotypes
rownames(Gsnp)=levels(phenodf$Variety)
colnames(Gsnp)=levels(phenodf$Variety)

#MTMs <- mmes(Phenos ~ trait:Rep,
#random= ~ vsm(usm(trait),ism(Variety)) ,
#rcov= ~ vsm(usm(trait),ism(id)),
#data=phenodfMTM, verbose = TRUE)
#summary(MTMs)

MTM <- mmer(cbind(P1, P2) ~ 1 + Rep,
random= ~ vsr(Variety, Gtc=unsm(2), Gu=Gsnp) ,
rcov= ~ vsr(units, Gtc=unsm(2)),
data=phenodf, verbose = TRUE)
summary(MTM)

P1MTM=rep(0,120)
P2MTM=rep(0,120)

for(i in c(1:120)){

  P1MTM[i]=MTM$U$`u:Variety`$P1[[i]]
  P2MTM[i]=MTM$U$`u:Variety`$P2[[i]]

}




In [ ]:
UniP1 <- mmes(P1 ~ 1 + Rep,
random= ~ vsm(ism(Variety), Gu=Gsnp) ,
rcov= ~ units,
data=phenodf, verbose = TRUE)
summary(UniP1)


In [ ]:
UniP2 <- mmes(P2 ~ 1 + Rep,
random= ~ vsm(ism(Variety), Gu=Gsnp) ,
rcov= ~ units,
data=phenodf, verbose = TRUE)
summary(UniP2)


**Calculate the correlations between the true genetic values and the BLUPs form the multi-trait and univariate models**

In [ ]:
print("MTMP1, True_BV" )
cor(P1MTM,phenodat$Genetic_Effect1[1:120])

print("MTMP2, True_BV" )
cor(P2MTM,phenodat$Genetic_Effect2[1:120])

print("UniP1, True_BV" )
cor(UniP1$u[,1],phenodat$Genetic_Effect1[1:120])

print("UniP2, True_BV" )
cor(UniP2$u[,1],phenodat$Genetic_Effect2[1:120])

**Was there a benefit to running the multi-trait model? Why?**

## Lab Questions

**Not all phenotypes cost the same to measure. Let's assume on of the phenotypes is much more costly/difficult to measure than the other. In this scenario we can afford to collect more data points for the less expensive phenotype.**

1) Run a multi-trait model using P1 and P2NA, and run a univariate model using P2NA. Provide a summary of the results from both models (**3 pts**).

2) Calculate the correlation between the BLUPs obtained from the multi-trait model (P2NA) and the true genetic values for phenotype 2. Compare this to the correlation of the BLUPs from univariate model (P2NA) and the true genetic values for phenotype 2. Do you get higher accuracy when utilizing information form phenotype 1 with a multi-trait model? (**1 pt**).

3) Run a multi-trait model using P1NA and P2, and run a univariate model using P1NA. Provide a summary of the results from both models (**3 pts**).

4) Calculate the correlation between the BLUPs obtained from the multi-trait model (P1NA) and the true genetic values for phenotype 2. Compare this to the correlation of the BLUPs from univaraite model (P1NA) and the true genetic values for phenotype 1. Do you get higher accuracy when utilizing information form phenotype 1 using a multi-trait model? (**1 pt**).

5) Calculate the correlations when using only the BLUPs from varieties that have no phenotypes for the trait of interest (P1NA and P2NA). How does this compare to correlations when using all BLUPs? (**1 pt**)

6) Based on these results, under what scenarios would you expect to see the largest gains in accuracy from using a multi-trait model? Why? (**1 pts**)